# Sumabasto · PROY · Catálogo revisado
Cuatro productos y dos bloques de 52 semanas de datos simulados. Los 30 casos quedan reservados para verificar la aplicación en el paso 4.

In [ ]:
%pip install -q scikit-learn==1.7.2 pandas==2.3.3 numpy==2.2.6 scipy==1.15.3 joblib==1.5.2

## Catálogo, simulación y partición temporal
Las variables de historial usan semanas anteriores. Una unidad corresponde a la presentación del producto.

In [ ]:
from collections import defaultdict
from datetime import date, timedelta
from random import Random
import pandas as pd
PRODUCTOS = ["CARTAVIO-50KG", "COSTENO-EXTRA-50KG", "FARAON-AZUL-50KG", "PARAMONGA-RUBIA-50KG"]
MARCAS = ["Cartavio", "Costeño", "Faraón Azul", "Paramonga"]
# APF2, Anexo G, Tabla 19. Cada perfil corresponde a un arquetipo sintético.
# Tupla: sacos mínimo/máximo, días mínimo/máximo. No se vinculan cuentas ni identidades.
FICHAS = [(5,5,9,9), (4,4,7,7), (6,6,12,12), (5,5,10,10),
          (4,4,8,10), (5,5,10,10), (6,6,10,10), (7,8,12,14)]
INICIO = date(2023, 10, 16)
SEMANAS = 156
VERSION = "sumabasto-apf3-calibrado-v2"
CAMPOS = ["ultima", "media_3", "media_8", "al_corte"]

def simular(semilla, anticipacion=.45, retiro=.06, pesos=(1,1,1,1), *,
            productos=PRODUCTOS, inicio=INICIO, semanas=SEMANAS):
    rng = Random(semilla)
    eventos = []
    cierres, iniciales = defaultdict(int), defaultdict(int)
    for puesto, (min_sacos, max_sacos, min_dias, max_dias) in enumerate(FICHAS):
        dia = rng.randrange(max_dias)
        while dia < semanas * 7:
            semana = dia // 7
            producto = rng.choices(range(len(productos)), weights=pesos, k=1)[0]
            sacos = rng.randint(min_sacos, max_sacos)
            anticipado = rng.random() < anticipacion
            decide_retiro = rng.random()
            retirado = anticipado and decide_retiro < retiro
            clave = (puesto, producto, semana)
            iniciales[clave] += sacos if anticipado else 0
            cierres[clave] += 0 if retirado else sacos
            eventos.append({"puesto": f"P{puesto+1:02}", "producto": productos[producto],
                            "fecha_objetivo": inicio + timedelta(days=dia),
                            "semana": inicio + timedelta(weeks=semana), "sacos": sacos,
                            "al_corte": anticipado, "retirado_antes_del_cierre": retirado,
                            "origen": "simulado"})
            dia += rng.randint(min_dias, max_dias)
    filas = []
    for semana in range(8, semanas):
        for producto, nombre in enumerate(productos):
            for puesto in range(8):
                historia = [cierres[puesto, producto, s] for s in range(semana-8, semana)]
                filas.append({"semana": inicio + timedelta(weeks=semana), "producto": nombre,
                              "puesto": f"P{puesto+1:02}", "ultima": historia[-1],
                              "media_3": sum(historia[-3:])/3, "media_8": sum(historia)/8,
                              "al_corte": iniciales[puesto, producto, semana],
                              "sacos_al_cierre": cierres[puesto, producto, semana], "origen": "simulado"})
    return pd.DataFrame(filas), pd.DataFrame(eventos)

def calidad(tabla):
    columnas = CAMPOS + ["sacos_al_cierre"]
    return {"filas": len(tabla), "duplicados": int(tabla.duplicated(["semana","producto","puesto"]).sum()),
            "nulos": int(tabla.isna().any(axis=1).sum()),
            "negativos": int((tabla[columnas] < 0).any(axis=1).sum()),
            "sobre_8": int((tabla[columnas] > 8).any(axis=1).sum())}




In [ ]:
import argparse
from datetime import date, timedelta
from pathlib import Path
from random import Random

import pandas as pd



CATALOGO = (
    ("COSTENO-EXTRA-50KG", "Arroz", "Costeño", "Extra, saco de 50 kg"),
    ("CARTAVIO-50KG", "Azúcar", "Cartavio", "Saco de 50 kg"),
    ("DON-VITTORIO-FIDEOS-500G", "Fideos", "Don Vittorio", "Espagueti, paquete de 500 g"),
    ("COSTENO-LENTEJAS-500G", "Lentejas", "Costeño", "Bolsa de 500 g"),
)
INICIO = date(2024, 1, 1)
CORTE = INICIO + timedelta(weeks=52)
SEMILLA = 20261126


def reservar_casos(prueba):
    rng = Random(SEMILLA + 1)
    casos = []
    for indice, (sku, *_resto) in enumerate(CATALOGO):
        filas = prueba[prueba.producto == sku]
        semanas = sorted(filas.semana.unique())
        cantidad = 8 if indice < 2 else 7
        for turno in range(cantidad):
            tramo = semanas[52 * turno // cantidad:52 * (turno + 1) // cantidad]
            semana = rng.choice(tramo)
            grupo = filas[filas.semana == semana]
            rol = "tienda" if (indice + turno) % 2 == 0 else "coordinacion"
            if rol == "tienda":
                con_demanda = grupo[grupo.sacos_al_cierre > 0]
                opciones = con_demanda if turno % 3 != 0 and not con_demanda.empty else grupo
                grupo = opciones.iloc[[rng.randrange(len(opciones))]]
            casos.append({
                "caso": f"SIM-{len(casos) + 1:02d}",
                "rol": rol,
                "semana": semana,
                "producto": sku,
                "puestos": "|".join(grupo.puesto),
                "unidades_al_corte_esperadas": int(grupo.al_corte.sum()),
                "unidades_al_cierre_simuladas": int(grupo.sacos_al_cierre.sum()),
                "comportamiento_esperado": "Consultar pronóstico autorizado y registrar solicitud",
                "resultado_obtenido": "",
                "evidencia": "",
                "estado": "pendiente",
                "origen": "simulado",
            })
    return pd.DataFrame(casos)


def preparar(destino):
    catalogo = pd.DataFrame(CATALOGO, columns=["producto", "categoria", "marca", "presentacion"])
    filas, eventos = simular(SEMILLA, productos=tuple(catalogo.producto), inicio=INICIO, semanas=104)
    desarrollo = filas[filas.semana < CORTE]
    prueba = filas[filas.semana >= CORTE]
    casos = reservar_casos(prueba)
    assert len(desarrollo) == 44 * 32 and len(prueba) == 52 * 32
    assert all(valor == 0 for clave, valor in calidad(filas).items() if clave != "filas")
    assert eventos.semana.nunique() == 104
    assert len(casos) == 30 and casos.rol.value_counts().to_dict() == {"tienda": 15, "coordinacion": 15}
    assert not casos.duplicated(["semana", "producto"]).any()
    destino.mkdir(parents=True, exist_ok=True)
    for nombre, tabla in {
        "catalogo": catalogo,
        "eventos_simulados": eventos,
        "desarrollo": desarrollo,
        "prueba_reservada": prueba,
        "casos_reservados": casos,
    }.items():
        tabla.to_csv(destino / f"{nombre}.csv", index=False, lineterminator="\n")
    print({"eventos": len(eventos), "desarrollo": len(desarrollo), "prueba": len(prueba),
           "casos": len(casos), "calidad": calidad(filas)})



In [ ]:
datos = Path("sumabasto_datos")
preparar(datos)
pd.read_csv(datos / "catalogo.csv")

## Ajuste de Ridge y evaluación reservada
Se elige alpha por MAE en cinco divisiones temporales del primer bloque. El segundo bloque compara Ridge con la regla al corte, en general y por producto.

In [ ]:
"""PROY: protocol frozen before evaluation; calibration and external real validation remain separate."""
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
FEATURES = ["producto"] + CAMPOS
Y = "sacos_al_cierre"
VERSION = "sumabasto-proy-candidato-v1"
def prep(model, productos=PRODUCTOS):
    return Pipeline([("preprocesamiento", ColumnTransformer([
        ("producto", OneHotEncoder(categories=[productos], handle_unknown="error", sparse_output=False), ["producto"]),
        ("volumen", StandardScaler(), CAMPOS)])), ("modelo", model)])
def predict(model, data):
    return np.maximum(0, model.predict(data[FEATURES]))
def mae(data, pred):
    return float(mean_absolute_error(data[Y], pred))
def metrics(data, pred):
    return dict(MAE=mae(data,pred), RMSE=float(mean_squared_error(data[Y],pred)**.5),
                R2=float(r2_score(data[Y],pred)), WAPE=float(100*np.abs(data[Y]-pred).sum()/data[Y].sum()))


In [ ]:
"""Ajusta Ridge en desarrollo y evalúa una sola vez en el bloque reservado."""

import argparse
import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit



def evaluar(datos, salida):
    desarrollo = pd.read_csv(datos / "desarrollo.csv")
    productos = pd.read_csv(datos / "catalogo.csv").producto.tolist()
    semanas = sorted(desarrollo.semana.unique())
    assert len(semanas) == 44
    cv = []
    for alpha in (1, 10, 100):
        errores = []
        for anteriores, siguientes in TimeSeriesSplit(n_splits=5).split(semanas):
            ajuste = desarrollo[desarrollo.semana <= semanas[anteriores[-1]]]
            validacion = desarrollo[desarrollo.semana.isin([semanas[i] for i in siguientes])]
            modelo = prep(Ridge(alpha=alpha), productos).fit(ajuste[FEATURES], ajuste[Y])
            errores.append(mae(validacion, predict(modelo, validacion)))
        cv.append(dict(alpha=alpha, MAE=float(np.mean(errores)), errores=errores))
    alpha = min(cv, key=lambda fila: (fila["MAE"], fila["alpha"]))["alpha"]
    modelo = prep(Ridge(alpha=alpha), productos).fit(desarrollo[FEATURES], desarrollo[Y])

    prueba = pd.read_csv(datos / "prueba_reservada.csv")
    assert desarrollo.semana.max() < prueba.semana.min() and prueba.semana.nunique() == 52
    assert set(prueba.producto) == set(desarrollo.producto) == set(productos)
    prueba = prueba.assign(ridge=predict(modelo, prueba), regla_al_corte=prueba.al_corte)
    resultados = []
    for producto, filas in [("general", prueba), *prueba.groupby("producto", sort=False)]:
        for nombre in ("regla_al_corte", "ridge"):
            resultados.append(dict(producto=producto, modelo=nombre, filas=len(filas),
                                   **metrics(filas, filas[nombre].to_numpy())))
    principal = min(resultados[:2], key=lambda fila: fila["MAE"])["modelo"]

    salida.mkdir(parents=True, exist_ok=True)
    artefacto = salida / "modelo_ridge.joblib"
    joblib.dump(modelo, artefacto)
    np.testing.assert_array_equal(predict(joblib.load(artefacto), prueba), prueba.ridge.to_numpy())
    pd.DataFrame(resultados).to_csv(salida / "metricas.csv", index=False)
    prueba.to_csv(salida / "predicciones.csv", index=False)
    informe = {
        "version": "sumabasto-proy-catalogo-24-meses-v1",
        "alpha": alpha, "validacion_temporal": cv, "estimacion_principal": principal,
        "sha256": hashlib.sha256(artefacto.read_bytes()).hexdigest(),
        "origen": "simulado", "metricas": resultados,
    }
    (salida / "evaluacion.json").write_text(json.dumps(informe, indent=2), encoding="utf-8")
    print(pd.DataFrame(resultados).to_string(index=False))
    print(f"alpha={alpha}, estimación principal={principal}")



In [ ]:
salida = Path("sumabasto_evaluacion")
evaluar(datos, salida)
pd.read_csv(salida / "metricas.csv")

## Decisión operativa
El menor MAE general decide la estimación principal. Con esta simulación se mantiene la regla al corte y Ridge queda como referencia. La validación de la aplicación sigue en el paso 4.